# Homework 3 — Data Preparation (6 Points)

**Dataset:** [BBBC010](https://bbbc.broadinstitute.org/BBBC010) — 100 wells of *C. elegans*
worms imaged in two channels (w1, w2), plus per-well binary foreground masks.

**Tasks**
1. Download the three BBBC010 archives.
2. Extract and inspect the file structure.
3. Pair each well's **binary mask** with its **w2 image** (channel 2).
4. Binarize masks, normalise 16→8-bit, convert to RGB, resize to **512×512**.
5. 80/20 split (seed=42), save in pix2pixHD layout (`train_A/train_B`, `test_A/test_B`).

`A` = mask (input), `B` = image (target) — same convention as Unit 5 pix2pixHD.

In [1]:
import os, re, glob, random, urllib.request, zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("Libraries ready.")

Libraries ready.


## 1. Download (3 zips)

In [2]:
data_dir = "data_download"
os.makedirs(data_dir, exist_ok=True)

urls = {
    "BBBC010_v2_images.zip":              "https://data.broadinstitute.org/bbbc/BBBC010/BBBC010_v2_images.zip",
    "BBBC010_v1_foreground.zip":          "https://data.broadinstitute.org/bbbc/BBBC010/BBBC010_v1_foreground.zip",
    "BBBC010_v1_foreground_eachworm.zip": "https://data.broadinstitute.org/bbbc/BBBC010/BBBC010_v1_foreground_eachworm.zip",
}

for fname, url in urls.items():
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        print(f"  {fname} already present ({os.path.getsize(fpath)//1024} KB)")
        continue
    print(f"  downloading {fname} ...")
    urllib.request.urlretrieve(url, fpath)
    print(f"    done ({os.path.getsize(fpath)//1024} KB)")

  downloading BBBC010_v2_images.zip ...
    done (66786 KB)
  downloading BBBC010_v1_foreground.zip ...
    done (1072 KB)
  downloading BBBC010_v1_foreground_eachworm.zip ...
    done (2670 KB)


## 2. Extract & inspect

Each archive contains a flat list of files.

In [3]:
for fname in urls:
    with zipfile.ZipFile(os.path.join(data_dir, fname), "r") as z:
        for member in z.namelist():
            if "__MACOSX" in member or "/.svn" in member or member.endswith(".svn-base"):
                continue
            z.extract(member, data_dir)

# Image files live flat in data_download/ alongside the zip; mask files live under
# data_download/BBBC010_v1_foreground/.
img_files  = sorted(glob.glob(os.path.join(data_dir, "*_w*.tif")))
mask_files = sorted(glob.glob(os.path.join(data_dir, "BBBC010_v1_foreground", "*_binary.png")))
print(f"Images: {len(img_files)}   masks: {len(mask_files)}")

Images: 200   masks: 100


## 3. Pair by well

Each image filename embeds the well code (e.g. `..._A02_w1_<uuid>.tif`). Each mask is
`<WELL>_binary.png`. We extract the well code, group images by well + channel, and
keep wells that have both a w2 image and a mask.

In [ ]:
WELL_RE = re.compile(r"_([A-P]\d{2})_w(\d)_")

well_to_w2 = {}    # well -> w2 image path
for f in img_files:
    m = WELL_RE.search(os.path.basename(f))
    if not m: continue
    well, ch = m.group(1), m.group(2)
    # TODO: keep only channel 2 images (the brightfield channel)
    if ch == ___:
        well_to_w2[well] = f

well_to_mask = {os.path.basename(f).replace("_binary.png", ""): f for f in mask_files}

common = sorted(set(well_to_w2) & set(well_to_mask))
print(f"Wells with both w2 image and mask: {len(common)}")
print(f"Example: well {common[0]}")
print(f"  image: {os.path.basename(well_to_w2[common[0]])}")
print(f"  mask:  {os.path.basename(well_to_mask[common[0]])}")

## 4. Visualise one raw pair

In [ ]:
sample = common[0]
img  = Image.open(well_to_w2[sample])
mask = Image.open(well_to_mask[sample])
print(f"image: {img.size} {img.mode}   mask: {mask.size} {mask.mode}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.array(img), cmap="gray"); ax[0].set_title(f"Well {sample} — w2 image")
ax[1].imshow(np.array(mask), cmap="gray"); ax[1].set_title("Foreground mask")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Process & save in pix2pixHD layout

* **Image**: 16-bit → 8-bit (min/max normalise), RGB, resize 512×512 (bilinear)
* **Mask**: binarize (>0 → 255), RGB, resize 512×512 (nearest)
* 80/20 split (`random.seed(42)`).

```
datasets/bbbc010_pix2pixhd/
├── train_A/   # masks  (80)
├── train_B/   # images (80)
├── test_A/
└── test_B/
```

In [ ]:
out_root = "datasets/bbbc010_pix2pixhd"
for split in ["train", "test"]:
    for sub in ["A", "B"]:
        os.makedirs(os.path.join(out_root, f"{split}_{sub}"), exist_ok=True)

# TODO: shuffle with seed 42 and pick the first 80 wells for training
random.seed(___)
random.shuffle(common)
N_TRAIN = ___
train = common[:N_TRAIN]
test  = common[N_TRAIN:]
print(f"train={len(train)}  test={len(test)}")

# TODO: target image size — pix2pixHD expects 512x512
TARGET = (___, ___)

def to_uint8_rgb(arr):
    arr = arr.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return (arr * 255).astype(np.uint8)

def process(well, split):
    # B: image
    img = np.array(Image.open(well_to_w2[well]))
    img_rgb = Image.fromarray(to_uint8_rgb(img)).convert("RGB").resize(TARGET, Image.BILINEAR)
    img_rgb.save(os.path.join(out_root, f"{split}_B", well + ".png"))
    # A: mask
    mask = np.array(Image.open(well_to_mask[well]).convert("L"))
    # TODO: binarize the mask — any pixel >0 becomes foreground (255), else 0
    binary = ((mask > ___).astype(np.uint8) * ___)
    Image.fromarray(binary).convert("RGB").resize(TARGET, Image.NEAREST).save(
        os.path.join(out_root, f"{split}_A", well + ".png"))

for w in train: process(w, "train")
for w in test:  process(w, "test")
print("Saved.")

## 6. Verify output

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(6, 8))
train_A = sorted(os.listdir(os.path.join(out_root, "train_A")))
for row, fname in enumerate(train_A[:3]):
    a = Image.open(os.path.join(out_root, "train_A", fname))
    b = Image.open(os.path.join(out_root, "train_B", fname))
    axes[row, 0].imshow(a, cmap="gray"); axes[row, 0].set_title(f"Mask (A) — {fname[:3]}")
    axes[row, 1].imshow(b, cmap="gray"); axes[row, 1].set_title("Image (B)")
    for ax in axes[row]: ax.axis("off")
plt.suptitle("Train samples — 512×512", fontsize=13)
plt.tight_layout(); plt.show()

for split in ["train", "test"]:
    na = len(os.listdir(os.path.join(out_root, f"{split}_A")))
    nb = len(os.listdir(os.path.join(out_root, f"{split}_B")))
    print(f"{split}: A={na}, B={nb}")

## Takeaway

Data is ready at `datasets/bbbc010_pix2pixhd/`. Next: `01_TrainModel.ipynb`.